In [38]:
import requests
url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet"

filename = "yellow_tripdata_2025-11.parquet" 

def download_file(url, filename):
     with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(filename, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192): 
                f.write(chunk)


In [ ]:
download_file(url, filename)
print("Download complete!")

In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark import SparkFiles



In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('pyspark-assignment') \
    .config("spark.sql.repl.eagerEval.enabled", True) \
    .getOrCreate()

  #.master("spark://172.19.0.2:7077")\

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/14 14:21:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
spark_df = spark.read.parquet("yellow_tripdata_2025-11.parquet")
spark_df.show(2)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [5]:
spark_df = spark_df.repartition(4)

In [7]:
spark_df.write.parquet('output/',mode="overwrite")

In [8]:
spark_df.show(5)

[Stage 12:==========================================================(8 + 0) / 8]

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       2| 2025-11-02 08:11:08|  2025-11-02 08:15:21|              1|         1.24|         1|                 N|         186|    

In [10]:
spark_df.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee',
 'cbd_congestion_fee']

In [ ]:
spark_df.withColumn("tpep_pickup_datetime",F.to_date("tpep_pickup_datetime")) \
        .select("tpep_pickup_datetime").show(5)

In [13]:
spark_df.withColumn("tpep_pickup_datetime",F.to_date("tpep_pickup_datetime")) \
        .where(F.col("tpep_pickup_datetime") == '2025-11-15').count()

162604

In [14]:
spark_df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [31]:
(spark_df.withColumn("trip_duration",(F.col("tpep_dropoff_datetime").cast("timestamp").cast("long") -
                                                F.col("tpep_pickup_datetime").cast("timestamp").cast("long")) / 60)  
        .select("trip_duration", "tpep_pickup_datetime",  "tpep_dropoff_datetime").show(5)
)

+-----------------+--------------------+---------------------+
|    trip_duration|tpep_pickup_datetime|tpep_dropoff_datetime|
+-----------------+--------------------+---------------------+
|              7.0| 2025-11-03 19:01:48|  2025-11-03 19:08:48|
|             17.4| 2025-11-10 12:46:18|  2025-11-10 13:03:42|
|8.533333333333333| 2025-11-02 14:17:02|  2025-11-02 14:25:34|
|7.733333333333333| 2025-11-08 20:05:28|  2025-11-08 20:13:12|
|9.483333333333333| 2025-11-02 15:30:08|  2025-11-02 15:39:37|
+-----------------+--------------------+---------------------+
only showing top 5 rows



In [37]:
(spark_df.withColumn("trip_duration_hrs",(F.col("tpep_dropoff_datetime").cast("timestamp").cast("long") -
                                                F.col("tpep_pickup_datetime").cast("timestamp").cast("long")) / 3600)  
        .select(F.max("trip_duration_hrs").alias("longest trip in hrs")).show()
)

[Stage 47:>                                                         (0 + 4) / 4]

+-------------------+
|longest trip in hrs|
+-------------------+
|  90.64666666666666|
+-------------------+



In [39]:

zone_lookup_url ='https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv'
zone_lookup= "zone_lookup"
download_file(zone_lookup_url, zone_lookup)
print("Download complete!")

Download complete!


In [6]:
zone_df = spark.read.csv("zone_lookup",header = True, inferSchema=True)
zone_df.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [8]:
zone_df.createOrReplaceTempView("zone_table")

In [11]:
spark.sql("""
SELECT * from zone_table
LIMIT 5
""").show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+



In [12]:
spark_df.createOrReplaceTempView("nov_2025_tripdata")

In [14]:
spark.sql("""
SELECT tpep_pickup_datetime, tpep_dropoff_datetime, trip_distance, PULocationID from nov_2025_tripdata
LIMIT 5
""").show()

+--------------------+---------------------+-------------+------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|PULocationID|
+--------------------+---------------------+-------------+------------+
| 2025-11-01 00:13:25|  2025-11-01 00:13:25|         1.68|          43|
| 2025-11-01 00:49:07|  2025-11-01 01:01:22|         2.28|         142|
| 2025-11-01 00:07:19|  2025-11-01 00:20:41|          2.7|         163|
| 2025-11-01 00:00:00|  2025-11-01 01:01:03|        12.87|         138|
| 2025-11-01 00:18:50|  2025-11-01 00:49:32|          8.4|         138|
+--------------------+---------------------+-------------+------------+



In [21]:
spark.sql("""
SELECT PULocationID, Zone, tpep_pickup_datetime as min_pu  
from nov_2025_tripdata a , zone_table B
where A.PULocationID = B.LocationID
AND tpep_pickup_datetime = (SELECT MIN(tpep_pickup_datetime) as min_pu  
from nov_2025_tripdata a)
""").show()

+------------+-----------+-------------------+
|PULocationID|       Zone|             min_pu|
+------------+-----------+-------------------+
|         132|JFK Airport|2008-12-31 23:04:21|
+------------+-----------+-------------------+



In [29]:
spark.sql("""
select a.*, b.zone from (
SELECT PULocationID,  count(1) as cnt 
from nov_2025_tripdata a
GROUP BY PULocationID  
order by cnt asc
limit 20 ) a, zone_table b
where A.PULocationID = B.LocationID
order by cnt asc
""").show()

+------------+---+--------------------+
|PULocationID|cnt|                zone|
+------------+---+--------------------+
|           5|  1|       Arden Heights|
|          84|  1|Eltingville/Annad...|
|         105|  1|Governor's Island...|
|         187|  3|       Port Richmond|
|         109|  4|         Great Kills|
|         111|  4| Green-Wood Cemetery|
|         199|  4|       Rikers Island|
|         204|  4|   Rossville/Woodrow|
|           2|  5|         Jamaica Bay|
|         251| 12|         Westerleigh|
|          59| 14|        Crotona Park|
|         172| 14|New Dorp/Midland ...|
|         176| 14|             Oakwood|
|         245| 14|       West Brighton|
|         253| 15|       Willets Point|
|          27| 16|Breezy Point/Fort...|
|         206| 17|Saint George/New ...|
|          30| 18|       Broad Channel|
|         156| 21|     Mariners Harbor|
|         118| 22|Heartland Village...|
+------------+---+--------------------+



In [ ]:
spark.sql("""

SELECT (
SELECT PULocationID,  MIN(tpep_pickup_datetime) as min_pu  
from nov_2025_tripdata a
GROUP BY PULocationID ) AS A,   zone_table B
where A.PULocationID = B.LocationID
""").show()